# SwipeSense: High-Accuracy Engagement Prediction and Relationship Outcome Analysis

This notebook studies user behavior in a synthetic dating app dataset. The main machine learning task is to predict a user's engagement segment (`app_usage_time_label`) using demographic, profile, behavioral, and app-usage signals. The secondary analysis explores whether engagement segments are meaningfully associated with dating outcomes such as mutual matches, ghosting, ignored chats, and formed relationships.

**Performance-oriented setup:** `app_usage_time_min` is included as a feature. This should produce much stronger prediction results because `app_usage_time_label` is a binned version of usage time. `match_outcome` is still excluded from prediction features and used only for post-model relationship analysis.

## 1. Research Questions

1. How well can we predict dating app engagement level when the direct usage-time field is included?
2. Which allowed machine learning model performs best in a performance-oriented setup?
3. Are engagement segments associated with match outcomes, or are relationship outcomes weakly connected to usage intensity?

This notebook is designed as a high-accuracy benchmark. Because it includes `app_usage_time_min`, the result should be interpreted as performance-oriented rather than a strict no-leakage estimate.

## 2. Setup

In [ ]:
# If you run this notebook in a fresh local environment, install the common ML stack first:
# %pip install pandas numpy scikit-learn matplotlib seaborn scipy

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")

RANDOM_STATE = 42
DEFAULT_DATA_PATH = Path("data/dating_app_behavior_dataset.csv")
DATA_PATH = DEFAULT_DATA_PATH

ENGAGEMENT_ORDER = [
    "Barely",
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Addicted",
    "Extreme User",
]

### Google Colab Dataset Upload

When this notebook is opened in Google Colab, upload `dating_app_behavior_dataset.csv` if the dataset is not already available at `data/dating_app_behavior_dataset.csv`. Uploading the Kaggle `.zip` file is also supported.

In [ ]:
def resolve_data_path(default_path):
    """Use the local dataset path, or ask for a CSV/ZIP upload in Google Colab."""
    if default_path.exists():
        print(f"Using local dataset: {default_path}")
        return default_path

    existing_csvs = sorted(Path(".").glob("*.csv"))
    if existing_csvs:
        print(f"Using CSV found in notebook directory: {existing_csvs[0]}")
        return existing_csvs[0]

    try:
        from google.colab import files
    except ModuleNotFoundError as exc:
        raise FileNotFoundError(
            "Dataset not found. Place dating_app_behavior_dataset.csv at "
            f"{default_path}, or upload a CSV file in the notebook directory."
        ) from exc

    print("Dataset not found. Please upload dating_app_behavior_dataset.csv or the Kaggle ZIP file.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No dataset file was uploaded.")

    uploaded_names = list(uploaded.keys())
    uploaded_csvs = [name for name in uploaded_names if name.lower().endswith(".csv")]
    if uploaded_csvs:
        data_path = Path(uploaded_csvs[0])
        print(f"Using uploaded CSV: {data_path}")
        return data_path

    uploaded_zips = [name for name in uploaded_names if name.lower().endswith(".zip")]
    if uploaded_zips:
        import zipfile

        extract_dir = Path("uploaded_dataset")
        extract_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(uploaded_zips[0]) as zip_ref:
            csv_members = [name for name in zip_ref.namelist() if name.lower().endswith(".csv")]
            if not csv_members:
                raise ValueError("The uploaded ZIP file does not contain a CSV file.")
            zip_ref.extract(csv_members[0], extract_dir)
            data_path = extract_dir / csv_members[0]
        print(f"Using CSV extracted from ZIP: {data_path}")
        return data_path

    raise ValueError("Please upload a .csv file or a .zip file that contains a CSV dataset.")


DATA_PATH = resolve_data_path(DEFAULT_DATA_PATH)

## 3. Load and Inspect the Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())

In [ ]:
required_columns = {
    "gender",
    "sexual_orientation",
    "location_type",
    "income_bracket",
    "education_level",
    "interest_tags",
    "app_usage_time_min",
    "app_usage_time_label",
    "swipe_right_ratio",
    "swipe_right_label",
    "likes_received",
    "mutual_matches",
    "profile_pics_count",
    "bio_length",
    "message_sent_count",
    "emoji_usage_rate",
    "last_active_hour",
    "swipe_time_of_day",
    "match_outcome",
}

missing_columns = required_columns.difference(df.columns)
assert not missing_columns, f"Missing expected columns: {sorted(missing_columns)}"

overview = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": [str(dtype) for dtype in df.dtypes],
        "missing_values": df.isna().sum().values,
        "unique_values": df.nunique().values,
    }
)
display(overview)

In [ ]:
duplicate_rows = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_rows:,}")
display(df.isna().sum().to_frame("missing_values"))

## 4. Exploratory Data Analysis

The target variable is imbalanced: `Extreme User` is the largest segment. Because of this, accuracy alone is not enough; macro F1 is important because it gives equal weight to smaller engagement classes.

In [ ]:
target_counts = df["app_usage_time_label"].value_counts().reindex(ENGAGEMENT_ORDER)
target_percent = (target_counts / len(df) * 100).round(2)

display(pd.DataFrame({"count": target_counts, "percent": target_percent}))

plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="app_usage_time_label", order=ENGAGEMENT_ORDER)
plt.title("Distribution of Engagement Segments")
plt.xlabel("Engagement segment")
plt.ylabel("Number of users")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
outcome_counts = df["match_outcome"].value_counts()
outcome_percent = (outcome_counts / len(df) * 100).round(2)

display(pd.DataFrame({"count": outcome_counts, "percent": outcome_percent}))

plt.figure(figsize=(11, 5))
sns.countplot(data=df, y="match_outcome", order=outcome_counts.index)
plt.title("Distribution of Match Outcomes")
plt.xlabel("Number of users")
plt.ylabel("Match outcome")
plt.tight_layout()
plt.show()

In [ ]:
numeric_columns = [
    "app_usage_time_min",
    "swipe_right_ratio",
    "likes_received",
    "mutual_matches",
    "profile_pics_count",
    "bio_length",
    "message_sent_count",
    "emoji_usage_rate",
    "last_active_hour",
]

display(df[numeric_columns].describe().T.round(3))

### Target Signal Check: Why `app_usage_time_min` Improves Performance

`app_usage_time_label` is a binned version of `app_usage_time_min`. Keeping the minute-level field gives the model direct access to the signal used to create the target. This is useful for a high-accuracy benchmark, but it should be explained clearly in the report.

In [ ]:
usage_label_ranges = (
    df.groupby("app_usage_time_label")["app_usage_time_min"]
    .agg(["count", "min", "max", "mean"])
    .reindex(ENGAGEMENT_ORDER)
    .round(2)
)

display(usage_label_ranges)

## 5. Feature Engineering and Preprocessing

`interest_tags` contains multiple comma-separated interests per user. It is converted into multi-hot columns. Categorical columns are one-hot encoded, and numeric columns are scaled inside the model pipeline.

The excluded fields are:

- `match_outcome`: reserved for relationship outcome analysis
- `app_usage_time_label`: prediction target

`app_usage_time_min` is intentionally included in this version to test the maximum achievable prediction performance.

In [ ]:
target_col = "app_usage_time_label"
excluded_cols = [target_col, "match_outcome"]

interest_dummies = df["interest_tags"].fillna("").str.get_dummies(sep=", ")
interest_dummies = interest_dummies.add_prefix("interest_")

model_df = pd.concat([df.drop(columns=["interest_tags"]), interest_dummies], axis=1)

X = model_df.drop(columns=excluded_cols)
y = model_df[target_col]

excluded_check = set(excluded_cols).intersection(X.columns)
assert not excluded_check, f"Excluded columns are still in X: {sorted(excluded_check)}"
assert "app_usage_time_min" in X.columns, "app_usage_time_min should be included in this high-accuracy version."

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()

print(f"Feature matrix shape: {X.shape}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Numeric / multi-hot features: {len(numeric_features)}")
print("Excluded from prediction:", excluded_cols)
print("Included direct usage-time feature: app_usage_time_min")

display(X.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {X_train.shape[0]:,}")
print(f"Testing rows: {X_test.shape[0]:,}")

split_balance = pd.DataFrame(
    {
        "train_percent": y_train.value_counts(normalize=True).reindex(ENGAGEMENT_ORDER) * 100,
        "test_percent": y_test.value_counts(normalize=True).reindex(ENGAGEMENT_ORDER) * 100,
    }
).round(2)
display(split_balance)

In [ ]:
def make_one_hot_encoder():
    """Return a OneHotEncoder that works across older and newer scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", make_one_hot_encoder(), categorical_features),
    ]
)

## 6. Baseline Model

The majority-class baseline predicts the most common engagement segment for every user. Any useful model should be compared against this baseline.

In [ ]:
majority_label = y_train.mode()[0]
baseline_pred = np.repeat(majority_label, len(y_test))

baseline_scores = {
    "model": "Majority baseline",
    "accuracy": accuracy_score(y_test, baseline_pred),
    "macro_f1": f1_score(y_test, baseline_pred, average="macro"),
    "weighted_f1": f1_score(y_test, baseline_pred, average="weighted"),
}

display(pd.DataFrame([baseline_scores]).round(4))

## 7. Train and Compare Allowed Models

The assignment guideline allows Logistic Regression, Decision Tree, Random Forest, SVM, and ANN/MLP for classification. All models below use the same preprocessing pipeline and the same train/test split.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "SVM (LinearSVC)": LinearSVC(
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        random_state=RANDOM_STATE,
    ),
    "MLP / ANN": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        alpha=0.001,
        max_iter=150,
        early_stopping=False,
        random_state=RANDOM_STATE,
    ),
}


def evaluate_predictions(model_name, y_true, y_pred):
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }


base_results = [baseline_scores]
fitted_models = {}
test_predictions = {"Majority baseline": baseline_pred}

for name, estimator in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", estimator),
        ]
    )
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    fitted_models[name] = pipeline
    test_predictions[name] = y_pred
    base_results.append(evaluate_predictions(name, y_test, y_pred))

base_results_df = pd.DataFrame(base_results).sort_values(
    by=["macro_f1", "accuracy"], ascending=False
)
display(base_results_df.round(4))

## 8. Classification Report and Confusion Matrix

The confusion matrix helps identify whether the model is learning minority segments or mostly predicting the dominant class.

In [ ]:
best_base_model_name = base_results_df.iloc[0]["model"]
best_base_pred = test_predictions[best_base_model_name]

print(f"Best base model by macro F1: {best_base_model_name}")
print(classification_report(y_test, best_base_pred, labels=ENGAGEMENT_ORDER, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, best_base_pred, labels=ENGAGEMENT_ORDER)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=ENGAGEMENT_ORDER,
    yticklabels=ENGAGEMENT_ORDER,
)
plt.title(f"Confusion Matrix: {best_base_model_name}")
plt.xlabel("Predicted engagement segment")
plt.ylabel("Actual engagement segment")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Lightweight Hyperparameter Tuning

Tuning is performed on a stratified subset of the training set to keep the notebook practical. The tuned models are evaluated on the untouched test set.

In [ ]:
tune_size = min(12000, len(X_train))

if tune_size < len(X_train):
    X_tune, _, y_tune, _ = train_test_split(
        X_train,
        y_train,
        train_size=tune_size,
        random_state=RANDOM_STATE,
        stratify=y_train,
    )
else:
    X_tune, y_tune = X_train, y_train

print(f"Tuning rows: {len(X_tune):,}")

tuning_jobs = {
    "Random Forest Tuned": RandomizedSearchCV(
        estimator=Pipeline(
            steps=[
                ("preprocess", preprocessor),
                (
                    "model",
                    RandomForestClassifier(
                        class_weight="balanced_subsample",
                        n_jobs=-1,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        param_distributions={
            "model__n_estimators": [100, 150, 250],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_leaf": [1, 5, 15],
        },
        n_iter=5,
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "SVM Tuned": GridSearchCV(
        estimator=Pipeline(
            steps=[
                ("preprocess", preprocessor),
                (
                    "model",
                    LinearSVC(
                        class_weight="balanced",
                        max_iter=5000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        param_grid={"model__C": [0.1, 1.0, 3.0]},
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
    ),
    "MLP / ANN Tuned": GridSearchCV(
        estimator=Pipeline(
            steps=[
                ("preprocess", preprocessor),
                (
                    "model",
                    MLPClassifier(
                        max_iter=120,
                        early_stopping=False,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        param_grid={
            "model__hidden_layer_sizes": [(64,), (64, 32)],
            "model__alpha": [0.0001, 0.001],
        },
        scoring="f1_macro",
        cv=2,
        n_jobs=-1,
    ),
}

tuned_results = []
tuned_models = {}
tuned_predictions = {}

for name, search in tuning_jobs.items():
    print(f"Tuning {name}...")
    search.fit(X_tune, y_tune)
    y_pred = search.best_estimator_.predict(X_test)

    tuned_models[name] = search.best_estimator_
    tuned_predictions[name] = y_pred

    row = evaluate_predictions(name, y_test, y_pred)
    row["best_params"] = search.best_params_
    tuned_results.append(row)

tuned_results_df = pd.DataFrame(tuned_results).sort_values(
    by=["macro_f1", "accuracy"], ascending=False
)
display(tuned_results_df.round(4))

In [ ]:
comparison_df = pd.concat(
    [
        base_results_df.assign(stage="base"),
        tuned_results_df.drop(columns=["best_params"]).assign(stage="tuned"),
    ],
    ignore_index=True,
).sort_values(by=["macro_f1", "accuracy"], ascending=False)

display(comparison_df.round(4))

plt.figure(figsize=(11, 5))
sns.barplot(
    data=comparison_df,
    x="macro_f1",
    y="model",
    hue="stage",
    dodge=False,
)
plt.title("Model Comparison by Macro F1")
plt.xlabel("Macro F1")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

## 10. Engagement Segment vs. Match Outcome

This section uses `match_outcome` only for analysis, not as a prediction feature. The goal is to understand whether engagement intensity is associated with relationship outcomes.

In [ ]:
outcome_crosstab = pd.crosstab(df["app_usage_time_label"], df["match_outcome"]).reindex(ENGAGEMENT_ORDER)
outcome_pct = pd.crosstab(
    df["app_usage_time_label"],
    df["match_outcome"],
    normalize="index",
).reindex(ENGAGEMENT_ORDER) * 100

display(outcome_crosstab)
display(outcome_pct.round(2))

In [ ]:
plt.figure(figsize=(13, 6))
outcome_pct.plot(kind="bar", stacked=True, figsize=(13, 6), colormap="tab20")
plt.title("Match Outcome Composition by Engagement Segment")
plt.xlabel("Engagement segment")
plt.ylabel("Percentage within engagement segment")
plt.xticks(rotation=30, ha="right")
plt.legend(title="Match outcome", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(13, 5))
sns.heatmap(outcome_pct, annot=True, fmt=".1f", cmap="YlGnBu")
plt.title("Normalized Match Outcome Heatmap by Engagement Segment")
plt.xlabel("Match outcome")
plt.ylabel("Engagement segment")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
chi2, p_value, dof, expected = chi2_contingency(outcome_crosstab)
n = outcome_crosstab.to_numpy().sum()
rows, cols = outcome_crosstab.shape
cramers_v = np.sqrt((chi2 / n) / min(rows - 1, cols - 1))

print(f"Chi-square statistic: {chi2:.3f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.6f}")
print(f"Cramer's V: {cramers_v:.4f}")

if cramers_v < 0.10:
    print("Interpretation: the association is very weak in practical terms.")
elif cramers_v < 0.30:
    print("Interpretation: the association is small to moderate.")
else:
    print("Interpretation: the association is relatively strong.")

## 11. Conclusion

This notebook uses a performance-oriented prediction setup. The direct usage-time field, `app_usage_time_min`, is included even though it strongly defines the target label. Because of that, models such as Decision Tree and Random Forest are expected to perform much better than in the strict no-leakage version. The result should therefore be presented as a high-accuracy benchmark, not as proof that indirect behavioral or demographic features alone can predict engagement.

The relationship analysis adds the dating-theme insight. If the heatmap and Cramer's V show only weak differences across engagement groups, then heavier app usage does not necessarily translate into better romantic outcomes. This is an important ethical and analytical point: digital behavior signals can describe activity, but they may not reliably explain complex relationship outcomes such as ghosting, ignored chats, or successful relationships.

Overall, the project combines a supervised classification task with an interpretable relationship-outcome analysis. For a balanced presentation, compare this version with the strict no-leakage notebook: the strict version tests generalization from indirect signals, while this version demonstrates how much predictive power comes from direct usage time.